In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings

import gradio as gr

In [2]:
MODEL = "gpt-4.1-nano"
DB_NAME= "vector_db"
load_dotenv(override=True)


True

Connect to Chroma; use Hugging Face all- MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(model_name =MODEL, temperature=0)

In [5]:
retriever.invoke("Welcome to CARS24")
llm.invoke("Welcome to CARS24")


AIMessage(content='Hello! Welcome to CARS24. How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 12, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_62b483d6f3', 'id': 'chatcmpl-DMGRlaukTmoREuI0QWPNWxLqaFHVu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--a61c84c7-9826-4405-8244-7b481cb67b49-0', usage_metadata={'input_tokens': 12, 'output_tokens': 15, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
SYSTEM_PROMPT_TEMPLATE = """
CARS24 - Company Overview
CARS24 is an Indian online used car marketplace founded in 2015, headquartered in Gurgaon.
It is among the top 4 organised players in India's used car segment.
CARS24 operates in India, UAE, Thailand, and Australia.

BUYING A CAR
- Browse thousands of verified used cars online at cars24.com
- Every car undergoes a 300-point quality check
- Home test drive available in select cities
- Doorstep delivery after purchase
- 7-day / 30-day return policy depending on the plan
- Warranties up to 3 years or 45,000 km
- Exchange your old car and get discounts

SELLING A CAR
- Get an instant online valuation in 3 easy steps
- AI-powered pricing engine based on 10 lakh+ real transactions
- Free doorstep inspection — a CARS24 representative visits your home
- Sell within 24 hours
- 20,000+ verified dealers across 1,500+ cities bid on your car via live auction
- Instant bank transfer after price agreement
- Free RC transfer and paperwork support
- Seller Kavach policy protects seller from legal liabilities post-sale (challans, police complaints, accidents)

CAR LOANS (LOANS24)
- Car loans with low interest rates
- Loan approval in seconds, 100% digital process
- Loan disbursal within a day
- Tenure: 12 to 72 months
- Zero down payment options available
- Loans against cars also available
- NBFC licensed by Reserve Bank of India (received July 2019)
- Financing partner: Poonawalla Fincorp
- Example: ₹2,00,000 loan at 12% APR for 24 months = EMI of ₹9,415

VALUE ADDED SERVICES
- Pay RTO / traffic challans online
- RC transfer assistance
- FASTag purchase and recharge
- Car scrapping service
- RSA (Roadside Assistance) — 24x7 support
- PUC (Pollution Under Control) assistance
- Car maintenance services
- Driver hire service

CARS24 MOTO
- Launched May 2020
- Buy and sell used two-wheelers: bikes, mopeds, scooters

HOW TO SELL YOUR CAR — STEP BY STEP
1. Enter car details (registration number or make/model/year/km)
2. Get instant online valuation
3. Book free home inspection appointment
4. CARS24 expert visits, inspects, and live auction begins
5. Dealers from across India bid — best price determined by competition
6. Accept the price — payment transferred instantly to your bank account
7. Car picked up from your location
8. Free RC transfer and all paperwork handled by CARS24

CONTACT & SUPPORT
- Email: care@cars24.com
- Available via Cars24 app (Android & iOS)
- 230+ cities, 1500+ channel partners across India

CARS24 AUCTION MODEL
- Pan-India auction model: dealers from entire country participate, not just local
- Real-time live bidding ensures maximum competition and best price
- Backed by AI pricing engine with real-time market data
Context:

You are a helpful, friendly customer support assistant for CARS24 — India's leading online used car platform.
You help users with:
- Buying used cars (process, warranties, returns, test drives)
- Selling their car (valuation, inspection, payment, RC transfer)
- Car loans (LOANS24 — rates, eligibility, tenure)
- Value-added services (challans, RC transfer, FASTag, RSA)

Always be concise, polite, and informative.
If you don't know something, say: "I don't have that information right now. Please contact care@cars24.com for help."

Use the context below to answer the user's question:
Context:

{context}

"""


In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)

    context = "\n\n".join(doc.page_content for doc in docs)

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ])

    return response.content

In [12]:
# answer_question("What is the best car you have?",[])
gr.ChatInterface(fn=answer_question).launch()


c:\Users\tbfro\projects\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


🔄 Original: hi 
✨ Rewritten: How to sell my car on Cars24?
🔍 Expanded into 3 queries: ['How to list my car on Cars24 for sale', 'Steps to sell a vehicle on Cars24', 'Guide to selling my car through Cars24']
📚 Retrieved 0 unique chunks


Traceback (most recent call last):
  File "c:\Users\tbfro\projects\llm_engineering\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\tbfro\projects\llm_engineering\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\tbfro\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\tbfro\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 1621, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\tbfro\projects\llm_engineering\.venv\Lib\site-packages\gradio\utils.py", line 882, in async_wrapper